# Ejercicio Módulo 5 - Dataset Swiss
**Inteligencia Artificial - CEIA - FIUBA**

**Franco Marcelo Morero**

Para aprender sobre regresión, vamos a utilizar un dataset clásico llamado Swiss, que proviene originalmente del lenguaje R. Este dataset contiene datos socioeconómicos de 47 provincias suizas a fines del siglo XIX. Cada fila representa una provincia, y las variables reflejan características demográficas y sociales relevantes para ese contexto histórico.

## Variables

- `Location`: Provincia donde se midieron los datos.
- `Fertility`: Tasa de fertilidad (número promedio de hijos por mujer)
- `Agriculture`:` Porcentaje de hombres ocupados en agricultura
- `Examination`: Porcentaje de hombres que completaron exámenes de educación superior
- `Education`: Nivel promedio de educación (escala arbitraria)
- `Catholic`: Porcentaje de población católica
- `Infant.Mortality`: Tasa de mortalidad infantil (por cada 1000 nacidos vivos)

## Que queremos predecir?

Vamos a utilizar este dataset para predecir la tasa de fertilidad en cada provincia mediante diferentes métodos de regresión.

--- 

Siguiendo el procedimiento típico de Machine Learning, vamos a leer los datos y separarlos en los datasets de entrenamiento y testeo utilizando Scikit-Learn...

In [2]:
import pandas as pd

df = pd.read_csv("swiss.csv")

df.head()

,Location,Fertility,Agriculture,Examination,Education,Catholic,Infant.Mortality
0,Courtelary,80.2,17.0,15,12,9.96,22.2
1,Delemont,83.1,45.1,6,9,84.84,22.2
2,Franches-Mnt,92.5,39.7,5,5,93.40,20.2
3,Moutier,85.8,36.5,12,7,33.77,20.3
4,Neuveville,76.9,43.5,17,15,5.16,20.6


In [3]:
print(f"Tenemos {df.shape[0]} observaciones")

Tenemos 47 observaciones


Obtenemos la variable objetivo (`Fertility`) y, por otro lado, los atributos (quitamos `Location` ya que no es un atributo numérico relevante para la regresión)

In [4]:
X = df.drop(["Fertility", "Location"], axis=1)
y = df["Fertility"]

Dado que tenemos pocas observaciones, vamos a separar el dataset en un 50% para entrenamiento y 50% para testeo:

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

## Regresión lineal múltiple

Arranquemos la primera parte del ejercicio. Para eso, vamos a entrenar un modelo de regresión lineal múltiple usando todos los atributos. Para ello debes:

1. Escalar los atributos usando `StandardScaler`
2. Entrenar el modelo usando el dataset de entrenamiento.
3. Obtener las predicciones sobre el dataset de testeo.
4. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.

In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
##### COMPLETAR AQUI LO PEDIDO

numerical_columns = ['Agriculture', 'Examination', 'Education', 'Catholic', 'Infant.Mortality']

# Creamos el preprocesamiento para las columnas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns)
    ]
)

# Creamos el pipeline con preprocesamiento y modelo
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Agriculture', 'Examination',
                                                   'Education', 'Catholic',
                                                   'Infant.Mortality'])])),
                ('regressor', LinearRegression())])

In [7]:
y_pred = pipeline.predict(X_test)
y_pred

array([57.42828964, 80.37882378, 62.63618648, 79.20635709, 68.77733438,
       72.26124115, 60.70120492, 59.73885808, 59.64334643, 67.16700043,
       83.15634618, 74.87176854, 82.24565613, 70.50407832, 76.70971832,
       62.01720646, 53.30039283, 33.02145223, 73.79394031, 81.31637977,
       70.26989169, 68.59225665, 75.23585645, 46.0476763 ])

In [8]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("R-cuadrado en test:", round(r2, 3))
print("Error absoluto medio:", round(mae, 3))
print("Error cuadrático medio:", round(mse, 3))

R-cuadrado en test: 0.574
Error absoluto medio: 6.094
Error cuadrático medio: 64.457


## Modelo con regularización

Para mejorar nuestro modelo, vamos a explorar técnicas de regresión lineal con regularización, que nos permiten controlar el sobreajuste y seleccionar variables relevantes automáticamente.

Existen dos variantes muy populares:

- Una penaliza la suma de los cuadrados de los coeficientes (regularización L2).
- La otra penaliza la suma del valor absoluto de los coeficientes (regularización L1).

Ambas ayudan a mejorar la generalización, pero una de ellas además puede eliminar variables (coeficientes exactamente cero), lo que ayuda a identificar qué atributos son realmente importantes.

Tu tarea:

1. Elegí correctamente cuál de los dos métodos de regularización usar para este problema. 
    - Pista: Queremos que el modelo sea capaz de hacer una selección automática de variables, dejando fuera aquellas que no aportan.
2. Implementá un pipeline que incluya escalado y el modelo elegido.
3. Buscá automáticamente el mejor valor del hiperparámetro de regularización (alpha) usando validación cruzada usando 3-folds.
4. Entrená el modelo con los datos de entrenamiento y obtené las predicciones para el set de testeo.
5. Calcular las métricas MAE, MSE  y $R^2$, e imprimir los resultados.
6. Imprimí los coeficientes resultantes e identificá qué variables fueron eliminadas (coeficiente = 0).

In [9]:
import numpy as np

from sklearn.linear_model import LassoCV, RidgeCV

Paremos un momento para entender qué hacen LassoCV y RidgeCV antes de continuar con la resolución:

> Tanto `LassoCV` como `RidgeCV` son implementaciones de regresión lineal con regularización que incluyen la búsqueda automática del mejor hiperparámetro alpha mediante validación cruzada.
>
> Ambos métodos prueban distintos valores de alpha y eligen el que minimiza el error del modelo, facilitando el proceso de ajuste sin necesidad de una búsqueda manual.
>
> Internamente, utilizan la métrica del error cuadrático medio (MSE) para evaluar el rendimiento del modelo en cada fold de la validación cruzada.
>
> Por ejemplo, si llamás a RidgeCV(alphas=alphas, cv=5), se hará una validación cruzada de 5 folds utilizando los valores de alpha que vos le pases, y se seleccionará el que obtenga el menor MSE promedio.
> 
> Una vez elegido el mejor alpha, el modelo final se entrena con todos los datos de entrenamiento usando ese valor.

¡Listo! Con todo lo que vimos hasta ahora, ya estás en condiciones de resolver esta parte y completar los 6 puntos propuestos

In [10]:
alphas = np.logspace(-4, 1, 500)

##### COMPLETAR AQUI LO PEDIDO
numerical_columns = ['Agriculture', 'Examination', 'Education', 'Catholic', 'Infant.Mortality']

# Creamos el preprocesamiento para las columnas
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_columns)
    ]
)

# Creamos el pipeline con preprocesamiento y modelo
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', LassoCV(alphas=alphas, cv=3))
])
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['Agriculture', 'Examination',
                                                   'Education', 'Catholic',
                                                   'Infant.Mortality'])])),
                ('regressor',
                 LassoCV(alphas=array([1.00000000e-04, 1.02334021e-04, 1.04722519e-04, 1.07166765e-04,
       1.09668060e-04, 1.12227736e-04, 1.14847155e-04, 1.17527712e-04,
       1.20270833e-04, 1.23077980e-04, 1...
       5.88219040e+00, 6.01948197e+00, 6.15997796e+00, 6.30375315e+00,
       6.45088409e+00, 6.60144909e+00, 6.75552832e+00, 6.91320378e+00,
       7.07455942e+00, 7.23968114e+00, 7.40865683e+00, 7.58157646e+00,
       7.75853206e+00, 7.93961785e+00, 8.12493021e+00, 8.31456781e+00,
       8.50863158e+00, 8.70722485e+00, 8.91045332e+00, 9.11842520e+00,
       9.33125118e+00, 9.54904456e+00, 9.77192128e+00, 1.00000000e+01]),
                         cv=3))])

In [11]:
y_pred = pipeline.predict(X_test)

In [12]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print("R-cuadrado en test:", round(r2, 3))
print("Error absoluto medio:", round(mae, 3))
print("Error cuadrático medio:", round(mse, 3))

R-cuadrado en test: 0.576
Error absoluto medio: 5.995
Error cuadrático medio: 64.294


In [13]:
pipeline.named_steps['preprocessor'].get_feature_names_out()

array(['num__Agriculture', 'num__Examination', 'num__Education',
       'num__Catholic', 'num__Infant.Mortality'], dtype=object)

In [ ]:
print(f"Los valores de los coeficientes de Lasso son {np.round(pipeline.named_steps['regressor'].coef_, 2)}")

Los valores de los coeficientes de Lasso son [-0.   -1.76 -2.4   1.72  3.2 ]


## Comparación de modelos y conclusiones

Completá la siguiente tabla con las métricas obtenidas para cada uno de los modelos que entrenaste:

| Modelo                        |  MAE  |  MSE   | $R^2$ |
| ----------------------------- |  ---  | ------ | ----- |
| Regresión Lineal              | 6.094 | 64.457 | 0.574 |
| Modelo Regularizado (Lasso)   | 5.995 | 64.294 | 0.576 |

> ⚠️ Asegurate de cambiar el nombre del modelo `Modelo Regularizado (L1 o L2)` según el modelo que usaste (Lasso o Ridge).

### Justificación

**¿Cuál de los modelos te parece que tuvo un mejor desempeño general?**

Tené en cuenta las tres métricas al responder, y también pensá en la complejidad del modelo (por ejemplo, si eliminó variables innecesarias).

Escribí tu respuesta a continuación:

Tuvo un mejor desempeño la Regression de Lasso, porque elimino el atributo Agriculture, con lo cual llegamos a un modelo mas simple. Tambien se puede observar que tenemos un apenas mejor coeficiente de determinación $R^2$ y ademas el error sobre los datos es un poco menor, como se puede ver en la tabla para MAE y MSE